# AIT-ADS baseline comparison

Compares eleven system-task baseline families evaluated on one AIT-ADS scenario's train/test split -- the AIT-ADS counterpart to `notebooks/baselines/cscas_baseline_comparison.ipynb` (see `baselines/_ait_ads_data.py`, `baselines/ait_ads_rf.py`, `baselines/ait_ads_logreg.py`, `baselines/ait_ads_xgboost.py`, `baselines/ait_ads_bert.py`, `baselines/ait_ads_securebert.py`, `baselines/ait_ads_zeroshot.py`, `baselines/ait_ads_anomaly.py`, `baselines/ait_ads_anomaly_iforest.py`, `baselines/ait_ads_mining.py`, `baselines/ait_ads_mining_anomaly.py`, `baselines/ait_ads_mining_anomaly_iforest.py`):

1. **Base schema (RF)** -- 5-feature `base` schema, `RandomForestClassifier`
2. **Base schema (LogReg)** -- same, `StandardScaler` + `LogisticRegression`
3. **Base schema (XGBoost)** -- same, `XGBClassifier`
4. **BERT** -- alert_group tokens (`sig:`/`host:`/`short:`) serialized to text, fine-tuned DistilBERT
5. **SecureBERT 2.0** -- same tokens, fine-tuned SecureBERT 2.0 (ModernBERT architecture, domain-adapted)
6. **Zero-shot** -- same tokens serialized to a prompt, no fine-tuning
7. **Anomaly (OneClassSVM)** -- same 5-feature base schema, `OneClassSVM` fit on benign-only train rows -- see the exclusion paragraph below for why this one is included as its own method rather than folded into the other six.
8. **Anomaly (IsolationForest)** -- same 5-feature base schema, benign-only fit, second anomaly-detector model family alongside method 7 -- isolates model choice within the anomaly-detector family the same way LogReg/XGBoost do within the classifier family. Tree-based, so unlike OneClassSVM it needs no feature scaling.
9. **Base schema + Mining (RF / LogReg / XGBoost)** -- the same 5-feature base schema extended with symbolic features mined by the attribute-mining pipeline (contrast-set + decision-tree rules) on the identical train split, fit with all three tabular classifiers (the mining siblings of methods 1-3, and the AIT-ADS counterpart of CSCAS's `cscas_mining.py`). One attribute-mining pass per `(scenario, grouping)` feeds all three; results are saved as `ait_ads_mining_*` (RF), `ait_ads_mining_logreg_*` and `ait_ads_mining_xgboost_*`.
10. **Anomaly + Mining (OneClassSVM)** -- method 9's mined symbolic features, on top of the reduced base schema, `OneClassSVM` fit on benign-only train rows -- the mining sibling of method 7, and the most complete "system" scenario (mining + task, one-class) evaluated here.
11. **Anomaly + Mining (IsolationForest)** -- method 9's mined symbolic features on the reduced base schema, `IsolationForest` fit on benign-only train rows -- the IsolationForest sibling of method 10 (and of method 8), 5-seeded (`random_state=0..4`).

**Two training-pool conditions, not three**: random undersampling and class-weighted (natural-ratio). No "guided" condition here -- AIT-ADS has no SCAS-equivalent outlier signal to guide sampling with (CSCAS-only, see `baselines/_sampling.py`'s own docstring). Zero-shot and all four anomaly methods have no training-pool step, so none of the conditions apply to them. Zero-shot and the two **OneClassSVM** anomaly methods are single deterministic runs (OneClassSVM has no `random_state`; its per-seed sd is exactly 0); the two **IsolationForest** anomaly methods are seed-averaged over 5 seeds like the classifiers. Every anomaly method reports `auc` alongside precision/recall/f1, plus `workload_at_recall` -- precision / FP / analyst-workload-reduction at the threshold hitting each target recall -- since the default `nu`/`contamination` = 0.05 cut over-flags relative to the true attack prevalence.

All methods are **seed-averaged (`N_SEEDS = 5`)** except zero-shot and the two OneClassSVM anomaly methods (single deterministic runs -- zero-shot at temperature=0, OneClassSVM has no `random_state` to average over). The two IsolationForest anomaly methods run the same 5-seed protocol (`random_state=0..4`), same convention as the CSCAS scripts.

**All ten methods share the exact same split** -- every script calls `_ait_ads_data.load_ait_ads_baseline_split(scenario, grouping_method)` (the two mining scripts use `load_ait_ads_baseline_split_with_groups`, the same split plus the AlertGroup objects mining needs), so results are directly comparable across model families for a given `(SCENARIO, GROUPING_METHOD)`, not just similarly configured pipelines. This replaced an earlier approach that pulled RF/LogReg/XGBoost from a separate, fixed_window-only pipeline (`run_model_comparison_attribute.py`) -- that could only ever guarantee *similar* splits, not identical ones, and couldn't be extended to the other 4 grouping methods without duplicating this same split logic anyway.

Results are **per (grouping method, scenario)** -- unlike CSCAS (pregrouped), AIT-ADS alerts need a grouping step first, and which of the 5 grouping methods (`fixed_window`, `time_delta`, `cscas_grouping`, `alertbert`, `deepcase`) is used is itself an axis, not a fixed choice (see `baselines/_ait_ads_grouping.py`). Set `SCENARIO` and `GROUPING_METHOD` in the Settings cell below -- every table down to the anomaly operating-point section uses that one choice. The final **Per-scenario results** section ignores it and lays out every `(baseline, model, training, grouping method)` combination, one table per scenario.

**`harrison` and `santos` are excluded from this notebook's aggregate reporting
entirely** (the full-grid and per-scenario sections below cover 6 of AIT-ADS's 8
scenarios). Their chronological 70/30 split puts every attack-labelled alert_group
in the last 30% of the timeline regardless of grouping method, leaving train 100%
benign -- so 6 of the 11 baseline families above (methods 1-5, 9: every trainable
classifier/text model) cannot be fit at all, under any grouping. The remaining 5
families (zero-shot, method 6, and the four anomaly variants, 7/8/10/11) don't need
attack examples in train and do run -- but only meaningfully under the 3 standard
groupings (`fixed_window`/`time_delta`/`cscas_grouping`): `alertbert`/`deepcase`
results also exist for them (nothing in the code blocks it), but sit on a grouping
step whose quality was only ever checked on `fox`/`russellmitchell`
(`baselines/grouping/_setup.TEST_SCENARIOS`) -- and DeepCASE additionally crashes
outright on `santos` (fixed event vocabulary, unseen event types). With that much
of the grid structurally missing or unverified, `harrison`/`santos` aren't
comparable to the other 6 scenarios, so this notebook leaves them out rather than
reporting a mostly-blank row. (The scripts still produced real result files for
them where they could -- `results/ait_ads_*_{harrison,santos}_*.json` -- this is a
reporting choice, not a claim the data doesn't exist.)

**`alertbert` / `deepcase` grouping is reported only for `fox` and `russellmitchell`**
-- the two in-scope scenarios held out of the learned groupers' training. The
AlertBERT checkpoint (pretrained, inference-only) and the DeepCASE ContextBuilder
(retrained each run) were both fit on `shaw` / `wardbeck` / `wheeler` / `wilson` --
so those 4 are in-training-distribution leakage and the baseline scripts skip them
(`_ait_ads_grouping.LEAKAGE_SCENARIOS`). `fox`/`russellmitchell` are the only
in-scope scenarios where grouping quality was actually validated (see the
per-scenario section).

**Run the scripts first** to generate the `results/*.json` files this notebook reads:
```
cd src/thesis/baselines
python ait_ads_rf.py
python ait_ads_logreg.py
python ait_ads_xgboost.py
python ait_ads_bert.py
python ait_ads_securebert.py
OLLAMA_MODEL=llama3.1:8b python ait_ads_zeroshot.py
python ait_ads_anomaly.py
python ait_ads_anomaly_iforest.py
python ait_ads_mining.py          # fits RF + LogReg + XGBoost on the mined matrix
python ait_ads_mining_anomaly.py
python ait_ads_mining_anomaly_iforest.py
```

In [21]:
from __future__ import annotations

import json

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

from thesis.baselines._results import RESULTS_DIR, is_anomaly, is_zero_shot, load_baseline_results
from thesis.paths import CACHE_DIR

## Settings

Edit `SCENARIO` / `GROUPING_METHOD` (and `ZEROSHOT_MODEL_SLUGS` if the zero-shot sweep used a different model set) and re-run -- nothing past this cell needs to change.

In [22]:
SCENARIO = "fox"  # fox, russellmitchell, shaw, wardbeck, wheeler, wilson (harrison/santos
# excluded from aggregate reporting -- see the intro cell -- but still selectable here for
# ad-hoc single-scenario inspection of whatever results exist for them)
GROUPING_METHOD = "fixed_window"  # fixed_window, time_delta, cscas_grouping, alertbert, deepcase
# alertbert/deepcase are only valid for fox/harrison/russellmitchell/santos -- see the intro cell above.

# Zero-shot sweep -- OLLAMA_MODEL with ":"/"/" replaced by "-", the MODELS list in
# shell-scripts/baselines/run_ait_ads_zeroshot.sh. All four appear in every table;
# any whose results/*.json isn't present just render as blank rows.
ZEROSHOT_MODEL_SLUGS = ["llama3.1-8b", "llama3.1-70b", "qwen2.5-7b", "qwen2.5-72b"]
_ZS = [f"ait_ads_zeroshot_{GROUPING_METHOD}_{SCENARIO}_{s}" for s in ZEROSHOT_MODEL_SLUGS]

RESULT_NAMES = [
    f"ait_ads_rf_{GROUPING_METHOD}_{SCENARIO}",
    f"ait_ads_logreg_{GROUPING_METHOD}_{SCENARIO}",
    f"ait_ads_xgboost_{GROUPING_METHOD}_{SCENARIO}",
    f"ait_ads_bert_{GROUPING_METHOD}_{SCENARIO}",
    f"ait_ads_securebert_{GROUPING_METHOD}_{SCENARIO}",
    *_ZS,
    f"ait_ads_anomaly_{GROUPING_METHOD}_{SCENARIO}_ocsvm",
    f"ait_ads_anomaly_{GROUPING_METHOD}_{SCENARIO}_iforest",
    f"ait_ads_mining_{GROUPING_METHOD}_{SCENARIO}",
    f"ait_ads_mining_logreg_{GROUPING_METHOD}_{SCENARIO}",
    f"ait_ads_mining_xgboost_{GROUPING_METHOD}_{SCENARIO}",
    f"ait_ads_mining_anomaly_{GROUPING_METHOD}_{SCENARIO}_ocsvm",
    f"ait_ads_mining_anomaly_{GROUPING_METHOD}_{SCENARIO}_iforest",
]

LABELS = {
    f"ait_ads_rf_{GROUPING_METHOD}_{SCENARIO}": f"Base schema (RF, {GROUPING_METHOD})",
    f"ait_ads_logreg_{GROUPING_METHOD}_{SCENARIO}": f"Base schema (LogReg, {GROUPING_METHOD})",
    f"ait_ads_xgboost_{GROUPING_METHOD}_{SCENARIO}": f"Base schema (XGBoost, {GROUPING_METHOD})",
    f"ait_ads_bert_{GROUPING_METHOD}_{SCENARIO}": f"BERT (tokens as text, {GROUPING_METHOD})",
    f"ait_ads_securebert_{GROUPING_METHOD}_{SCENARIO}": f"SecureBERT 2.0 (tokens as text, {GROUPING_METHOD})",
    **{k: f"Zero-shot ({s}, {GROUPING_METHOD})" for k, s in zip(_ZS, ZEROSHOT_MODEL_SLUGS)},
    f"ait_ads_anomaly_{GROUPING_METHOD}_{SCENARIO}_ocsvm": f"Anomaly (OneClassSVM, benign-only, {GROUPING_METHOD})",
    f"ait_ads_anomaly_{GROUPING_METHOD}_{SCENARIO}_iforest": f"Anomaly (IsolationForest, benign-only, {GROUPING_METHOD})",
    f"ait_ads_mining_{GROUPING_METHOD}_{SCENARIO}": f"Base schema + Mining (RF, {GROUPING_METHOD})",
    f"ait_ads_mining_logreg_{GROUPING_METHOD}_{SCENARIO}": f"Base schema + Mining (LogReg, {GROUPING_METHOD})",
    f"ait_ads_mining_xgboost_{GROUPING_METHOD}_{SCENARIO}": f"Base schema + Mining (XGBoost, {GROUPING_METHOD})",
    f"ait_ads_mining_anomaly_{GROUPING_METHOD}_{SCENARIO}_ocsvm": f"Anomaly + Mining (OneClassSVM, benign-only, {GROUPING_METHOD})",
    f"ait_ads_mining_anomaly_{GROUPING_METHOD}_{SCENARIO}_iforest": f"Anomaly + Mining (IsolationForest, benign-only, {GROUPING_METHOD})",
}

COLORS = {
    f"ait_ads_rf_{GROUPING_METHOD}_{SCENARIO}": "#CCBB44",
    f"ait_ads_logreg_{GROUPING_METHOD}_{SCENARIO}": "#228833",
    f"ait_ads_xgboost_{GROUPING_METHOD}_{SCENARIO}": "#66CCEE",
    f"ait_ads_bert_{GROUPING_METHOD}_{SCENARIO}": "#EE6677",
    f"ait_ads_securebert_{GROUPING_METHOD}_{SCENARIO}": "#AA3377",
    **{k: c for k, c in zip(_ZS, ["#BBBBBB", "#999999", "#DDDDDD", "#777777"])},
    f"ait_ads_anomaly_{GROUPING_METHOD}_{SCENARIO}_ocsvm": "#44BB99",
    f"ait_ads_anomaly_{GROUPING_METHOD}_{SCENARIO}_iforest": "#117733",
    f"ait_ads_mining_{GROUPING_METHOD}_{SCENARIO}": "#EE8866",
    f"ait_ads_mining_logreg_{GROUPING_METHOD}_{SCENARIO}": "#DDAA77",
    f"ait_ads_mining_xgboost_{GROUPING_METHOD}_{SCENARIO}": "#CC9955",
    f"ait_ads_mining_anomaly_{GROUPING_METHOD}_{SCENARIO}_ocsvm": "#DDCC77",
    f"ait_ads_mining_anomaly_{GROUPING_METHOD}_{SCENARIO}_iforest": "#999933",
}

# Scoped per (scenario, grouping method) so different combinations' figures/summaries don't overwrite each other.
FIGURES_DIR = RESULTS_DIR / "figures" / SCENARIO / GROUPING_METHOD
FIGURES_DIR.mkdir(parents=True, exist_ok=True)

In [23]:
results = {}
for name in RESULT_NAMES:
    try:
        results[name] = load_baseline_results(name)
    except FileNotFoundError as e:
        print(f"[skip] {e}")

print(f"Loaded results for: {list(results.keys())}")

Loaded results for: ['ait_ads_rf_fixed_window_fox', 'ait_ads_logreg_fixed_window_fox', 'ait_ads_xgboost_fixed_window_fox', 'ait_ads_bert_fixed_window_fox', 'ait_ads_securebert_fixed_window_fox', 'ait_ads_zeroshot_fixed_window_fox_llama3.1-8b', 'ait_ads_zeroshot_fixed_window_fox_llama3.1-70b', 'ait_ads_zeroshot_fixed_window_fox_qwen2.5-7b', 'ait_ads_zeroshot_fixed_window_fox_qwen2.5-72b', 'ait_ads_anomaly_fixed_window_fox_ocsvm', 'ait_ads_anomaly_fixed_window_fox_iforest', 'ait_ads_mining_fixed_window_fox', 'ait_ads_mining_logreg_fixed_window_fox', 'ait_ads_mining_xgboost_fixed_window_fox', 'ait_ads_mining_anomaly_fixed_window_fox_ocsvm', 'ait_ads_mining_anomaly_fixed_window_fox_iforest']


In [24]:
def plot_baseline(condition_key: str, title: str) -> None:
    """Grouped bar chart across every loaded method that actually reports
    `condition_key` -- excludes zero-shot and anomaly always (both flat
    shape, no conditions -- see plot_zeroshot()/plot_anomaly() below).
    Saves to FIGURES_DIR/{condition_key}.png before displaying."""
    metrics = ["precision", "recall", "f1"]
    methods = [
        n for n in RESULT_NAMES
        if n in results and not is_zero_shot(results[n]) and not is_anomaly(results[n])
        and condition_key in results[n]
    ]
    if not methods:
        print(f"No results loaded with a '{condition_key}' condition -- run the baseline scripts first.")
        return

    x = np.arange(len(metrics))
    width = 0.85 / len(methods)

    fig, ax = plt.subplots(figsize=(10, 5))
    for i, name in enumerate(methods):
        values = [results[name][condition_key][m] for m in metrics]
        offset = (i - (len(methods) - 1) / 2) * width
        bars = ax.bar(
            x + offset, values, width,
            label=LABELS.get(name, name), color=COLORS.get(name),
        )
        ax.bar_label(bars, fmt="%.2f", fontsize=7, padding=2, rotation=90)

    ax.set_xticks(x)
    ax.set_xticklabels([m.capitalize() for m in metrics])
    ax.set_ylim(0, 1.2)
    ax.set_ylabel("Score")
    ax.set_title(f"{title} ({SCENARIO}, {GROUPING_METHOD})")
    ax.legend(loc="upper center", bbox_to_anchor=(0.5, -0.12), ncol=3, fontsize=8)
    ax.spines[["top", "right"]].set_visible(False)
    fig.tight_layout()

    save_path = FIGURES_DIR / f"{condition_key}.png"
    fig.savefig(save_path, dpi=150, bbox_inches="tight")
    print(f"Figure written to {save_path}")

    plt.show()

## Summary table

Tidy view of all loaded results for this scenario -- handy to copy straight into a slide.

In [25]:
CONDITION_LABELS = {
    "random": "Random undersampling",
    "class_weighted": "Class-weighted (natural-ratio)",
}

rows = []
for name, data in results.items():
    if is_zero_shot(data) or is_anomaly(data):
        rows.append({
            "method": LABELS.get(name, name),
            "condition": "(no training)",
            "auc": data.get("auc"),
            "precision": data["precision"],
            "recall": data["recall"],
            "f1": data["f1"],
        })
    else:
        for condition_key, condition_label in CONDITION_LABELS.items():
            if condition_key not in data:
                continue
            rows.append({
                "method": LABELS.get(name, name),
                "condition": condition_label,
                "auc": None,
                "precision": data[condition_key]["precision"],
                "recall": data[condition_key]["recall"],
                "f1": data[condition_key]["f1"],
            })

if not rows:
    print("No results loaded -- run the baseline scripts first.")
    summary_df = pd.DataFrame(columns=["method", "condition", "auc", "precision", "recall", "f1"])
else:
    summary_df = pd.DataFrame(rows).sort_values(["condition", "method"]).reset_index(drop=True)
    summary_csv_path = RESULTS_DIR / f"ait_ads_{SCENARIO}_{GROUPING_METHOD}_baseline_summary.csv"
    summary_df.to_csv(summary_csv_path, index=False)
    print(f"Summary written to {summary_csv_path}")
summary_df

Summary written to /Users/annavisman/stack/TUDelft/thesis/msc-thesis/src/thesis/baselines/results/ait_ads_fox_fixed_window_baseline_summary.csv


,method,condition,auc,precision,recall,f1
0,"Anomaly (IsolationForest, benign-only, fixed_w...",(no training),0.820226,0.542348,0.695575,0.609398
1,"Anomaly (OneClassSVM, benign-only, fixed_window)",(no training),0.895237,0.470414,0.703540,0.563830
2,"Anomaly + Mining (IsolationForest, benign-only...",(no training),0.801106,0.384265,0.411504,0.396915
3,"Anomaly + Mining (OneClassSVM, benign-only, fi...",(no training),0.775927,0.393316,0.676991,0.497561
4,"Zero-shot (llama3.1-70b, fixed_window)",(no training),NaN,0.352818,0.747788,0.479433
5,"Zero-shot (llama3.1-8b, fixed_window)",(no training),NaN,0.316116,0.676991,0.430986
6,"Zero-shot (qwen2.5-72b, fixed_window)",(no training),NaN,0.452316,0.734513,0.559865
7,"Zero-shot (qwen2.5-7b, fixed_window)",(no training),NaN,0.272871,0.765487,0.402326
8,"BERT (tokens as text, fixed_window)",Class-weighted (natural-ratio),NaN,1.000000,0.657522,0.793374
9,"Base schema (LogReg, fixed_window)",Class-weighted (natural-ratio),NaN,0.857955,0.668142,0.751244


## LaTeX summary table

One block per baseline, with the configuration knobs that vary across baselines
broken out as columns:

| knob | column | values |
|---|---|---|
| input representation | **Features** | `Base (5)` reduced tabular schema · `Base (5) + mined` reduced schema plus attribute-mined symbolic features · `tokens as text` the `sig:`/`host:`/`short:` tokens serialized to text/prompt |
| training regime | **Training** | one row per **sampling condition** (`random` undersampling · `class-weighted` natural-ratio) for the trainable classifiers; `benign-only` for the one-class anomaly detectors; `none` for the zero-shot LLMs |
| learner | **Model** | RF / LogReg / XGBoost / DistilBERT / SecureBERT 2.0 / the four zero-shot models (`ZEROSHOT_MODEL_SLUGS`) / OneClassSVM / IsolationForest |

**Precision / Recall / F1** are at each model's own decision threshold, shown as
mean$_{\pm\text{sd}}$ over the 5 seeds. The four zero-shot rows and the OneClassSVM
anomaly rows are single deterministic runs (OneClassSVM shown as $\pm$0.000); the
IsolationForest anomaly rows are 5-seeded like the classifiers. All rows are
scored on this scenario's full test split. No `guided` condition and no
paper-target row -- both are CSCAS-only (see the intro cell).

The zero-shot block lists all four models in `ZEROSHOT_MODEL_SLUGS`; a model whose
`results/*.json` isn't present for this `(scenario, grouping)` renders as a blank
row (run `shell-scripts/baselines/run_ait_ads_zeroshot.sh`).

Writes `results/ait_ads_{SCENARIO}_{GROUPING_METHOD}_baseline_summary.tex` (the
`\resizebox` in the output needs `\usepackage{graphicx}`). `table_rows` from this
cell feeds the alert-volume table below.

In [26]:
# One block per baseline, with the config knobs that vary across baselines
# (input representation / training regime / learner) broken out as columns.
# Mirrors the CSCAS notebook's LaTeX-summary cell, minus the paper-target row
# (CSCAS-only), with two training conditions instead of three, and the
# zero-shot block expanded to the full ZEROSHOT_MODEL_SLUGS sweep.
# `table_rows` is reused by the alert-volume cell.
#
#   kind "trainable" -> one row per sampling condition (random / class_weighted)
#   kind "flat"      -> single row from the top-level metrics (zero-shot / anomaly)
ZEROSHOT_DISPLAY = {  # pretty names for the LaTeX Model column; slug used as-is otherwise
    "llama3.1-8b": "Llama-3.1-8B", "llama3.1-70b": "Llama-3.1-70B",
    "qwen2.5-7b": "Qwen2.5-7B", "qwen2.5-72b": "Qwen2.5-72B",
}

TABLE_SPEC = [  # (section, baseline, model, features, kind, result_key, regime_label)
    ("internal", "Internal system",   "Logistic Regression", "Base (5)",         "trainable", f"ait_ads_logreg_{GROUPING_METHOD}_{SCENARIO}",        None),
    ("internal", "Internal system",   "Random Forest",       "Base (5)",         "trainable", f"ait_ads_rf_{GROUPING_METHOD}_{SCENARIO}",            None),
    ("internal", "Internal system",   "XGBoost",             "Base (5)",         "trainable", f"ait_ads_xgboost_{GROUPING_METHOD}_{SCENARIO}",       None),
    ("mining",   "Internal + mining", "Random Forest",       "Base (5) + mined", "trainable", f"ait_ads_mining_{GROUPING_METHOD}_{SCENARIO}",        None),
    ("mining",   "Internal + mining", "Logistic Regression", "Base (5) + mined", "trainable", f"ait_ads_mining_logreg_{GROUPING_METHOD}_{SCENARIO}", None),
    ("mining",   "Internal + mining", "XGBoost",             "Base (5) + mined", "trainable", f"ait_ads_mining_xgboost_{GROUPING_METHOD}_{SCENARIO}", None),
    *[
        ("llm", "LLM (zero-shot)", ZEROSHOT_DISPLAY.get(s, s), "tokens as text", "flat", k, "none")
        for s, k in zip(ZEROSHOT_MODEL_SLUGS, _ZS)
    ],
    ("llm",      "LLM (general)",         "DistilBERT",          "tokens as text",   "trainable", f"ait_ads_bert_{GROUPING_METHOD}_{SCENARIO}",       None),
    ("llm",      "LLM (domain-specific)", "SecureBERT 2.0",      "tokens as text",   "trainable", f"ait_ads_securebert_{GROUPING_METHOD}_{SCENARIO}", None),
    ("anom",     "Anomaly",               "OneClassSVM",         "Base (5)",         "flat",      f"ait_ads_anomaly_{GROUPING_METHOD}_{SCENARIO}_ocsvm",         "benign-only"),
    ("anom",     "Anomaly",               "IsolationForest",     "Base (5)",         "flat",      f"ait_ads_anomaly_{GROUPING_METHOD}_{SCENARIO}_iforest",       "benign-only"),
    ("anom",     "Anomaly + mining",      "OneClassSVM",         "Base (5) + mined", "flat",      f"ait_ads_mining_anomaly_{GROUPING_METHOD}_{SCENARIO}_ocsvm",  "benign-only"),
    ("anom",     "Anomaly + mining",      "IsolationForest",     "Base (5) + mined", "flat",      f"ait_ads_mining_anomaly_{GROUPING_METHOD}_{SCENARIO}_iforest", "benign-only"),
]

TRAIN_CONDITIONS = [
    ("random", "random"),
    ("class_weighted", "class-weighted"),
]


def _pick(src: dict | None) -> dict | None:
    if src is None:
        return None
    out = {m: src.get(m) for m in ("precision", "recall", "f1")}
    seeds = src.get("seeds") or []
    # A flat row with no seed list is the OneClassSVM detector (deterministic
    # convex fit -- its run-to-run sd is genuinely 0.000) or a zero-shot row
    # (single run, no sd). is_anomaly() tells them apart.
    zero_sd = src.get("kind") == "anomaly"
    for m in ("precision", "recall", "f1"):
        vals = [s[m] for s in seeds]
        if len(vals) > 1:
            out[f"{m}_sd"] = float(np.std(vals, ddof=1))
        else:
            out[f"{m}_sd"] = 0.0 if zero_sd else None
    return out


table_rows = []
for gid, (section, baseline, model, features, kind, key, regime) in enumerate(TABLE_SPEC):
    data = results.get(key)
    if kind == "flat":
        entries = [(regime, _pick(data))]
    else:  # trainable -- one entry per sampling condition
        entries = [
            (label, _pick(data.get(cond) if data else None))
            for cond, label in TRAIN_CONDITIONS
        ]
    for training_label, m in entries:
        row = {"gid": gid, "section": section, "Baseline": baseline,
               "Model": model, "Features": features, "Training": training_label}
        for col in ("Precision", "Recall", "F1"):
            row[col] = None if m is None else m[col.lower()]
            row[f"{col}_sd"] = None if m is None else m.get(f"{col.lower()}_sd")
        table_rows.append(row)

table_df = pd.DataFrame(table_rows).drop(columns=["gid", "section"]).round(4)


# --- emit LaTeX (repeated Baseline/Model/Features blanked within a block) ---
def _cell(v, sd=None) -> str:
    if v is None or (isinstance(v, float) and pd.isna(v)):
        return ""
    if sd is None or pd.isna(sd):
        return f"{v:.3f}"
    return f"${v:.3f}_{{\\pm {sd:.3f}}}$"


tex_lines = []
prev_section = prev_gid = None
for row in table_rows:
    if prev_section is not None and row["section"] != prev_section:
        tex_lines.append(r"\midrule")
    elif prev_gid is not None and row["gid"] != prev_gid:
        tex_lines.append(r"\addlinespace")
    first = row["gid"] != prev_gid
    prev_section, prev_gid = row["section"], row["gid"]
    b = row["Baseline"] if first else ""
    mo = row["Model"] if first else ""
    fe = row["Features"] if first else ""
    tex_lines.append(
        f"{b} & {mo} & {fe} & {row['Training']} & "
        f"{_cell(row['Precision'], row['Precision_sd'])} & "
        f"{_cell(row['Recall'], row['Recall_sd'])} & "
        f"{_cell(row['F1'], row['F1_sd'])}" + r" \\"
    )

_gm_tex = GROUPING_METHOD.replace("_", r"\_")
latex = (
    r"""\begin{table}[htbp]
\centering
\small
\caption{Summary of baseline performance on the AIT-ADS """ + f"{SCENARIO}" + r""" scenario,
grouping \texttt{""" + _gm_tex + r"""}. Trainable classifiers report both training-pool
sampling conditions (random undersampling, class-weighted natural-ratio) as
mean$_{\pm\text{sd}}$ over 5 seeds; the anomaly detectors are fit on benign rows
only and the zero-shot LLMs have no training step. Precision/recall/F1 are at each
model's own decision threshold. All rows are scored on this scenario's full test
split. The anomaly rows are at the detectors' default rejection rate ($\nu$ /
contamination $= 0.05$); Table~\ref{tab:ait-ads-""" + f"{SCENARIO}-{GROUPING_METHOD}" + r"""-anomaly-operating-points}
reports them at a tuned threshold. No \emph{guided} condition and no paper-target
row -- both are CSCAS-only.}
\label{tab:ait-ads-""" + f"{SCENARIO}-{GROUPING_METHOD}" + r"""-baseline-summary}
\resizebox{\textwidth}{!}{%
\begin{tabular}{llllccc}
\toprule
\textbf{Baseline} & \textbf{Model} & \textbf{Features} & \textbf{Training} & \textbf{Precision} & \textbf{Recall} & \textbf{F1} \\
\midrule
"""
    + "\n".join(tex_lines)
    + r"""
\bottomrule
\end{tabular}%
}
\end{table}
"""
)

latex_path = RESULTS_DIR / f"ait_ads_{SCENARIO}_{GROUPING_METHOD}_baseline_summary.tex"
latex_path.write_text(latex, encoding="utf-8")
print(f"LaTeX table written to {latex_path}\n")
print(latex)
table_df

LaTeX table written to /Users/annavisman/stack/TUDelft/thesis/msc-thesis/src/thesis/baselines/results/ait_ads_fox_fixed_window_baseline_summary.tex

\begin{table}[htbp]
\centering
\small
\caption{Summary of baseline performance on the AIT-ADS fox scenario,
grouping \texttt{fixed\_window}. Trainable classifiers report both training-pool
sampling conditions (random undersampling, class-weighted natural-ratio) as
mean$_{\pm\text{sd}}$ over 5 seeds; the anomaly detectors are fit on benign rows
only and the zero-shot LLMs have no training step. Precision/recall/F1 are at each
model's own decision threshold. All rows are scored on this scenario's full test
split. The anomaly rows are at the detectors' default rejection rate ($\nu$ /
contamination $= 0.05$); Table~\ref{tab:ait-ads-fox-fixed_window-anomaly-operating-points}
reports them at a tuned threshold. No \emph{guided} condition and no paper-target
row -- both are CSCAS-only.}
\label{tab:ait-ads-fox-fixed_window-baseline-summary}
\resize

,Baseline,Model,Features,Training,Precision,Precision_sd,Recall,Recall_sd,F1,F1_sd
0,Internal system,Logistic Regression,Base (5),random,0.8073,0.0940,0.6646,0.0037,0.7270,0.0350
1,Internal system,Logistic Regression,Base (5),class-weighted,0.8580,0.0000,0.6681,0.0000,0.7512,0.0000
2,Internal system,Random Forest,Base (5),random,0.4037,0.0049,0.7398,0.0145,0.5223,0.0059
3,Internal system,Random Forest,Base (5),class-weighted,0.4236,0.0029,0.6938,0.0037,0.5260,0.0026
4,Internal system,XGBoost,Base (5),random,0.4077,0.0082,0.7451,0.0202,0.5270,0.0113
5,Internal system,XGBoost,Base (5),class-weighted,0.3848,0.0000,0.6947,0.0000,0.4953,0.0000
6,Internal + mining,Random Forest,Base (5) + mined,random,0.3996,0.0118,0.7168,0.0133,0.5131,0.0111
7,Internal + mining,Random Forest,Base (5) + mined,class-weighted,0.4268,0.0093,0.6894,0.0020,0.5271,0.0071
8,Internal + mining,Logistic Regression,Base (5) + mined,random,0.8793,0.0199,0.6681,0.0000,0.7592,0.0074
9,Internal + mining,Logistic Regression,Base (5) + mined,class-weighted,0.8988,0.0000,0.6681,0.0000,0.7665,0.0000


## Alert volume / confusion counts

`precision` and `recall` become a concrete analyst load once the class balance is
fixed. This table covers **every row of the summary table above** (all sampling
conditions, all models). TP/FP/FN are on this scenario's full test split;
**Est. FP/day** projects the benign false-positive rate onto the test window's
wall-clock span.

The test-split composition and span are reconstructed **from the cached
`alert_groups`** (`CACHE_DIR/<scenario>/groups/<grouping_method>/alert_groups/alert_groups_raw.json`)
-- the same last-`TEST_FRAC` chronological slice `load_ait_ads_baseline_split`
takes, no pipeline re-run. `alertbert`/`deepcase` groupings whose cache isn't
present locally fall back to an anomaly result's confusion counts for the
composition, and `Est. FP/day` shows `--` (the span needs the grouped data). The
`always-alert` row is the ceiling. Writes
`results/ait_ads_{SCENARIO}_{GROUPING_METHOD}_alert_volume.tex`.

In [27]:
# Confusion counts + alert volume for every row of the summary table above,
# derived from its (precision, recall) and the test-split composition.
#
# The test-split composition and its wall-clock span are reconstructed from the
# cached alert_groups -- no pipeline re-run: read the cached AlertGroup list,
# keep the benign/attack-labelled ones, take the last TEST_FRAC in cache order
# (the cache is written start_ts-sorted). This is exactly the slice
# _ait_ads_data.load_ait_ads_baseline_split takes. alertbert/deepcase groupings
# whose cache isn't present locally fall back to an anomaly result's own
# tp/fp/tn/fn (its workload_at_recall) for the composition, and "Est. FP/day"
# is left blank (the span needs the grouped data).
TEST_FRAC = 0.3  # matches load_ait_ads_baseline_split's default


def _split_from_cache():
    raw = (
        CACHE_DIR / SCENARIO / "groups" / GROUPING_METHOD
        / "alert_groups" / "alert_groups_raw.json"
    )
    if not raw.exists():
        return None
    groups = json.loads(raw.read_text())
    labelled = [g for g in groups if g.get("group_label") in ("benign", "attack")]
    if not labelled:
        return None
    test = labelled[int(len(labelled) * (1 - TEST_FRAC)):]
    pos = sum(g["group_label"] == "attack" for g in test)
    neg = sum(g["group_label"] == "benign" for g in test)
    starts = [g["start_ts"] for g in test]
    ends = [g["end_ts"] if g.get("end_ts") is not None else g["start_ts"] for g in test]
    days = (max(ends) - min(starts)) / 86_400.0
    return pos, neg, (days if days > 0 else None), "cached alert_groups"


def _split_from_anomaly():
    for name, data in results.items():
        if not is_anomaly(data):
            continue
        for tgt in ("0.90", "0.95", "0.99"):
            wl = (data.get("workload_at_recall") or {}).get(tgt)
            if wl:
                return round(wl["tp"] + wl["fn"]), round(wl["fp"] + wl["tn"]), None, f"{name} workload_at_recall"
    return None


_info = _split_from_cache() or _split_from_anomaly()
_vol_cols = ["Baseline", "Model", "Training", "TP", "FP", "FN", "Precision", "Recall", "Est. FP/day"]
if _info is None or not table_rows:
    print("No test-split composition available -- need the cached alert_groups or a loaded anomaly result.")
    vol_df = pd.DataFrame(columns=_vol_cols)
else:
    TEST_POS, TEST_NEG, TEST_DAYS, _src = _info
    TEST_PREVALENCE = TEST_POS / (TEST_POS + TEST_NEG)
    _span = f"{TEST_DAYS:.2f} days" if TEST_DAYS else "timespan unavailable (grouping cache absent)"
    print(f"Test split: {TEST_POS} attack / {TEST_NEG} benign  |  {_span}  [{_src}]\n")

    def _counts(p, r):
        tp = round(r * TEST_POS)
        fn = TEST_POS - tp
        fp = TEST_NEG if not p else round(tp * (1 - p) / p)
        return tp, fp, fn

    def _fp_day(fp):
        return None if not TEST_DAYS else fp / TEST_DAYS

    vol_rows, vol_tex = [], []
    prev_section = prev_gid = None
    for row in table_rows:  # built by the LaTeX-summary cell above
        p, r = row["Precision"], row["Recall"]
        if p is None or r is None or (isinstance(r, float) and pd.isna(r)):
            continue
        if prev_section is not None and row["section"] != prev_section:
            vol_tex.append(r"\midrule")
        elif prev_gid is not None and row["gid"] != prev_gid:
            vol_tex.append(r"\addlinespace")
        first = row["gid"] != prev_gid
        prev_section, prev_gid = row["section"], row["gid"]

        tp, fp, fn = _counts(p, r)
        fpd = _fp_day(fp)
        vol_rows.append({"Baseline": row["Baseline"], "Model": row["Model"],
                         "Training": row["Training"], "TP": tp, "FP": fp, "FN": fn,
                         "Precision": round(p, 3), "Recall": round(r, 3),
                         "Est. FP/day": None if fpd is None else round(fpd, 1)})
        b = row["Baseline"] if first else ""
        mo = row["Model"] if first else ""
        vol_tex.append(
            f"{b} & {mo} & {row['Training']} & {tp} & {fp} & {fn} & "
            f"{p:.3f} & {r:.3f} & " + ("--" if fpd is None else f"{fpd:,.1f}") + r" \\"
        )

    # always-alert ceiling
    _tp, _fp, _fn = _counts(TEST_PREVALENCE, 1.0)
    _fpd = _fp_day(_fp)
    vol_rows.append({"Baseline": "always-alert", "Model": "--", "Training": "--",
                     "TP": _tp, "FP": _fp, "FN": _fn, "Precision": round(TEST_PREVALENCE, 3),
                     "Recall": 1.0, "Est. FP/day": None if _fpd is None else round(_fpd, 1)})
    vol_tex.append(r"\midrule")
    vol_tex.append(
        f"always-alert & -- & -- & {_tp} & {_fp} & {_fn} & "
        f"{TEST_PREVALENCE:.3f} & 1.000 & " + ("--" if _fpd is None else f"{_fpd:,.1f}") + r" \\"
    )

    vol_df = pd.DataFrame(vol_rows)

    _gm_tex = GROUPING_METHOD.replace("_", r"\_")
    _days_txt = f"{TEST_DAYS:.2f}" if TEST_DAYS else "n/a"
    _neg_txt = f"{TEST_NEG:,}".replace(",", "{,}")
    vol_latex = (
        r"""\begin{table}[htbp]
\centering
\small
\caption{Confusion-count and alert-volume view of every row in
Table~\ref{tab:ait-ads-""" + f"{SCENARIO}-{GROUPING_METHOD}" + r"""-baseline-summary} (AIT-ADS
"""
        + f"{SCENARIO}" + r""" scenario, grouping \texttt{""" + _gm_tex + r"""}). TP/FP/FN are on
this scenario's full test split (""" + f"{TEST_POS}" + r""" attack / """ + _neg_txt + r""" benign
alert\_groups) -- precision and recall projected onto that fixed composition.
\emph{Est.\ FP/day} scales the benign false-positive rate to the test window's
wall-clock span (${\sim}$""" + _days_txt + r""" days) -- the analyst-load axis F1 hides.
The \texttt{always-alert} row is the ceiling.}
\label{tab:ait-ads-""" + f"{SCENARIO}-{GROUPING_METHOD}" + r"""-alert-volume}
\resizebox{\textwidth}{!}{%
\begin{tabular}{lllrrrccr}
\toprule
\textbf{Baseline} & \textbf{Model} & \textbf{Training} & \textbf{TP} & \textbf{FP} & \textbf{FN} & \textbf{Precision} & \textbf{Recall} & \textbf{Est. FP/day} \\
\midrule
"""
        + "\n".join(vol_tex)
        + r"""
\bottomrule
\end{tabular}%
}
\end{table}
"""
    )

    vol_tex_path = RESULTS_DIR / f"ait_ads_{SCENARIO}_{GROUPING_METHOD}_alert_volume.tex"
    vol_tex_path.write_text(vol_latex, encoding="utf-8")
    print(f"Written to {vol_tex_path}\n")
    print(vol_latex)
vol_df

Test split: 226 attack / 2868 benign  |  1.47 days  [cached alert_groups]

Written to /Users/annavisman/stack/TUDelft/thesis/msc-thesis/src/thesis/baselines/results/ait_ads_fox_fixed_window_alert_volume.tex

\begin{table}[htbp]
\centering
\small
\caption{Confusion-count and alert-volume view of every row in
Table~\ref{tab:ait-ads-fox-fixed_window-baseline-summary} (AIT-ADS
fox scenario, grouping \texttt{fixed\_window}). TP/FP/FN are on
this scenario's full test split (226 attack / 2{,}868 benign
alert\_groups) -- precision and recall projected onto that fixed composition.
\emph{Est.\ FP/day} scales the benign false-positive rate to the test window's
wall-clock span (${\sim}$1.47 days) -- the analyst-load axis F1 hides.
The \texttt{always-alert} row is the ceiling.}
\label{tab:ait-ads-fox-fixed_window-alert-volume}
\resizebox{\textwidth}{!}{%
\begin{tabular}{lllrrrccr}
\toprule
\textbf{Baseline} & \textbf{Model} & \textbf{Training} & \textbf{TP} & \textbf{FP} & \textbf{FN} & \textbf{Pre

,Baseline,Model,Training,TP,FP,FN,Precision,Recall,Est. FP/day
0,Internal system,Logistic Regression,random,150,36,76,0.807,0.665,24.4
1,Internal system,Logistic Regression,class-weighted,151,25,75,0.858,0.668,17.0
2,Internal system,Random Forest,random,167,247,59,0.404,0.740,167.6
3,Internal system,Random Forest,class-weighted,157,214,69,0.424,0.694,145.2
4,Internal system,XGBoost,random,168,244,58,0.408,0.745,165.5
5,Internal system,XGBoost,class-weighted,157,251,69,0.385,0.695,170.3
6,Internal + mining,Random Forest,random,162,243,64,0.400,0.717,164.8
7,Internal + mining,Random Forest,class-weighted,156,210,70,0.427,0.689,142.5
8,Internal + mining,Logistic Regression,random,151,21,75,0.879,0.668,14.2
9,Internal + mining,Logistic Regression,class-weighted,151,17,75,0.899,0.668,11.5


## Per-scenario results (all applicable groupings)

One table per AIT-ADS scenario -- **6 of the 8** (`harrison`/`santos` excluded, see
the intro cell) -- every `(baseline, model, training condition, grouping method)` we
ran, with its **F1** (`CROSS_METRIC` → `precision` / `recall` and re-run). Reads
**every** `ait_ads_*` result JSON, ignoring the Settings cell.

**Applicable grouping methods:**

| grouping | applied to |
|---|---|
| `fixed_window`, `time_delta`, `cscas_grouping` | every in-scope scenario |
| `alertbert`, `deepcase` | **`fox` and `russellmitchell` only** |

`alertbert` / `deepcase` are restricted to `fox` / `russellmitchell` — the two
in-scope scenarios held out of the learned groupers' training:

- **AlertBERT** — a **pretrained** MLM checkpoint (`mlm_1l_4h_16d_original_1_60k`,
  `external/AlertBERT/saved_models`), used for **inference only**, never trained during a
  baseline run. That checkpoint was trained on `shaw` / `wardbeck` / `wheeler` / `wilson`.
  For e.g. `fox` + `alertbert`: `group_alerts_alertbert(fox_alerts, δ=1.5, θ=1024)` loads
  the checkpoint, embeds fox's alerts, clusters them; the classifier then trains/evals on
  those fox groups with the normal 70/30 split.
- **DeepCASE** — **no** pretrained checkpoint; its ContextBuilder is **trained fresh each
  run** on `shaw` / `wardbeck` / `wheeler` / `wilson` (`_setup.DEEPCASE_TRAIN_SCENARIOS`),
  then applied to group the target scenario.

So `shaw`/`wardbeck`/`wheeler`/`wilson` are in-distribution (leakage) for the learned
groupers; `fox`/`russellmitchell` are the only in-scope scenarios where grouping quality
was actually validated (`_setup.TEST_SCENARIOS`). The `δ`/`θ`/`context_length`/`eps`
operating points (`_ait_ads_grouping.ALERTBERT_BEST` / `DEEPCASE_BEST`) were also picked
on `fox` + `russellmitchell` group purity.

**Split**: fixed last-30%-of-timeline chronological cut (`test_frac = 0.3`); the column
shows `0.70 / 0.30` plus the train/test alert_group counts where the grouping cache is
local (the counts differ a lot by grouping method). Anomaly F1 is at the default `nu` /
`contamination` cut.

Writes `results/ait_ads_per_scenario_{CROSS_METRIC}.tex` — one `table` per scenario.

In [28]:
# Per-scenario view -- one table per AIT-ADS scenario: every (baseline, model,
# training condition, grouping method) that produced a result, with its
# CROSS_METRIC score. Reads EVERY ait_ads_* result JSON, ignoring the Settings
# cell. alertbert/deepcase are restricted to fox/russellmitchell (the only
# scenarios held out of the learned groupers' training -- see the markdown).
# Writes results/ait_ads_per_scenario_<metric>.tex (one \table per scenario;
# the \resizebox needs \usepackage{graphicx}, shrink-only).
from thesis.configs import load_scenarios

CROSS_METRIC = "f1"  # "f1" | "precision" | "recall" -- re-run for a different one
PS_DROP_EMPTY = False  # True -> omit grouping rows that produced no result
AIT_ADS_SCENARIOS = [s for s in load_scenarios("ait-ads") if s not in ("harrison", "santos")]
# harrison/santos excluded from all aggregate reporting -- see the intro cell's exclusion
# note: their chronological split leaves train 100% benign under every grouping method, so
# 6 of 11 baseline families can't be fit at all, and the remaining 5 are only trustworthy
# under 3 of 5 groupings -- too thin to compare against the other 6 scenarios.
STANDARD_GROUPINGS = ["fixed_window", "time_delta", "cscas_grouping"]
LEARNED_GROUPINGS = ["alertbert", "deepcase"]
LEARNED_OK = {"fox", "russellmitchell"}  # only these are held out of AlertBERT/DeepCASE training
TEST_FRAC = 0.3  # matches load_ait_ads_baseline_split

ZEROSHOT_DISPLAY = {
    "llama3.1-8b": "Llama-3.1-8B", "llama3.1-70b": "Llama-3.1-70B",
    "qwen2.5-7b": "Qwen2.5-7B", "qwen2.5-72b": "Qwen2.5-72B",
}

# (section, baseline, model, kind, key_template)  -- {g}=grouping, {s}=scenario
PS_SPEC = [
    ("internal", "Internal system",       "LogReg",          "trainable", "ait_ads_logreg_{g}_{s}"),
    ("internal", "Internal system",       "RF",              "trainable", "ait_ads_rf_{g}_{s}"),
    ("internal", "Internal system",       "XGBoost",         "trainable", "ait_ads_xgboost_{g}_{s}"),
    ("mining",   "Internal + mining",     "RF",              "trainable", "ait_ads_mining_{g}_{s}"),
    ("mining",   "Internal + mining",     "LogReg",          "trainable", "ait_ads_mining_logreg_{g}_{s}"),
    ("mining",   "Internal + mining",     "XGBoost",         "trainable", "ait_ads_mining_xgboost_{g}_{s}"),
    ("llm",      "LLM (general)",         "DistilBERT",      "trainable", "ait_ads_bert_{g}_{s}"),
    ("llm",      "LLM (domain-specific)", "SecureBERT 2.0",  "trainable", "ait_ads_securebert_{g}_{s}"),
    *[
        ("llm", "LLM (zero-shot)", ZEROSHOT_DISPLAY.get(sl, sl), "flat", "ait_ads_zeroshot_{g}_{s}_" + sl)
        for sl in ZEROSHOT_MODEL_SLUGS
    ],
    ("anom",     "Anomaly",               "OneClassSVM",     "flat", "ait_ads_anomaly_{g}_{s}_ocsvm"),
    ("anom",     "Anomaly",               "IsolationForest", "flat", "ait_ads_anomaly_{g}_{s}_iforest"),
    ("anom",     "Anomaly + mining",      "OneClassSVM",     "flat", "ait_ads_mining_anomaly_{g}_{s}_ocsvm"),
    ("anom",     "Anomaly + mining",      "IsolationForest", "flat", "ait_ads_mining_anomaly_{g}_{s}_iforest"),
]
PS_CONDITIONS = [("random", "random"), ("class_weighted", "class-weighted")]
_FLAT_TRAINING = {"llm": "none", "anom": "benign-only"}

_pscache: dict = {}


def _psload(key):
    if key not in _pscache:
        try:
            _pscache[key] = load_baseline_results(key)
        except FileNotFoundError:
            _pscache[key] = None
    return _pscache[key]


def _psval(key, kind, cond):
    d = _psload(key)
    if d is None:
        return None
    if kind == "flat":
        return d.get(CROSS_METRIC)
    sub = d.get(cond)
    return sub.get(CROSS_METRIC) if sub else None


def _applicable(scenario):
    return STANDARD_GROUPINGS + (LEARNED_GROUPINGS if scenario in LEARNED_OK else [])


def _split_str(scenario, grouping):
    raw = (
        CACHE_DIR / scenario / "groups" / grouping / "alert_groups" / "alert_groups_raw.json"
    )
    if not raw.exists():
        return "0.70 / 0.30"
    labelled = [
        x for x in json.loads(raw.read_text())
        if x.get("group_label") in ("benign", "attack")
    ]
    n = len(labelled)
    n_tr = int(n * (1 - TEST_FRAC))
    return f"0.70 / 0.30 ({n_tr} / {n - n_tr})"


per_scenario_blocks = []  # (scenario, DataFrame, groupings)
for s in AIT_ADS_SCENARIOS:
    groupings = _applicable(s)
    split = {g: _split_str(s, g) for g in groupings}
    rows = []
    for section, baseline, model, kind, tmpl in PS_SPEC:
        conds = [(None, _FLAT_TRAINING[section])] if kind == "flat" else PS_CONDITIONS
        for cond_key, cond_label in conds:
            scores = {g: _psval(tmpl.format(g=g, s=s), kind, cond_key) for g in groupings}
            if all(v is None for v in scores.values()):
                continue
            for g in groupings:
                if PS_DROP_EMPTY and scores[g] is None:
                    continue
                rows.append({
                    "section": section, "Baseline": baseline, "Model": model,
                    "Training": cond_label, "Grouping": g, "Split": split[g],
                    CROSS_METRIC.upper(): None if scores[g] is None else round(scores[g], 3),
                })
    if rows:
        per_scenario_blocks.append((s, pd.DataFrame(rows), groupings))


def _psc(v):
    return "" if v is None or (isinstance(v, float) and pd.isna(v)) else f"{v:.3f}"


metric_hdr = CROSS_METRIC.upper()
tex_tables = []
for s, df, groupings in per_scenario_blocks:
    body = []
    prev_section = prev_block = None
    for _, row in df.iterrows():
        block = (row["Baseline"], row["Model"], row["Training"])
        if prev_section is not None and row["section"] != prev_section:
            body.append(r"\midrule")
        elif prev_block is not None and block != prev_block:
            body.append(r"\addlinespace")
        first = block != prev_block
        prev_section, prev_block = row["section"], block
        b = row["Baseline"] if first else ""
        mo = row["Model"] if first else ""
        tr = row["Training"] if first else ""
        g_disp = row["Grouping"].replace("_", "\\_")
        body.append(
            f"{b} & {mo} & {tr} & {g_disp} & {row['Split']} & {_psc(row[metric_hdr])} \\\\"
        )
    header = " & ".join([
        "\\textbf{Baseline}", "\\textbf{Model}", "\\textbf{Training}",
        "\\textbf{Grouping}", "\\textbf{Split (tr/te)}", "\\textbf{" + metric_hdr + "}",
    ])
    caption = (
        "AIT-ADS " + metric_hdr + " for scenario \\texttt{" + s + "} -- every "
        "(baseline, model, training condition, grouping method) that produced a result. "
        "Split is the fixed last-30\\%-of-timeline chronological cut (train / test "
        "fraction, with alert\\_group counts where the grouping cache is local). "
        "alertbert / deepcase are restricted to fox / russellmitchell (held out of the "
        "learned groupers' training). Anomaly rows are at the default $\\nu$ / "
        "contamination cut."
    )
    tex_tables.append(
        "\\begin{table}[htbp]\n\\centering\n\\small\n"
        "\\caption{" + caption + "}\n"
        "\\label{tab:ait-ads-per-scenario-" + s + "-" + CROSS_METRIC + "}\n"
        "\\resizebox{\\ifdim\\width>\\linewidth\\linewidth\\else\\width\\fi}{!}{%\n"
        "\\begin{tabular}{lllllc}\n\\toprule\n"
        + header + " \\\\\n\\midrule\n"
        + "\n".join(body)
        + "\n\\bottomrule\n\\end{tabular}%\n}\n\\end{table}"
    )

if not per_scenario_blocks:
    print("No AIT-ADS results loaded -- run the baseline scripts / scp the results first.")
else:
    ps_path = RESULTS_DIR / f"ait_ads_per_scenario_{CROSS_METRIC}.tex"
    ps_path.write_text("\n\n".join(tex_tables), encoding="utf-8")
    print(f"Wrote {len(per_scenario_blocks)} tables ({CROSS_METRIC}) to {ps_path}\n")

    from IPython.display import display

    for s, df, groupings in per_scenario_blocks:
        print(f"=== {s}   (groupings: {', '.join(groupings)}, {len(df)} rows) ===")
        display(
            df[["Baseline", "Model", "Training", "Grouping", "Split", metric_hdr]]
            .set_index(["Baseline", "Model", "Training", "Grouping"])
        )

Wrote 6 tables (f1) to /Users/annavisman/stack/TUDelft/thesis/msc-thesis/src/thesis/baselines/results/ait_ads_per_scenario_f1.tex

=== fox   (groupings: fixed_window, time_delta, cscas_grouping, alertbert, deepcase, 120 rows) ===


Split  \
Baseline         Model           Training    Grouping                                    
Internal system  LogReg          random      fixed_window    0.70 / 0.30 (7216 / 3094)   
                                             time_delta      0.70 / 0.30 (5840 / 2504)   
                                             cscas_grouping  0.70 / 0.30 (4264 / 1828)   
                                             alertbert                     0.70 / 0.30   
                                             deepcase                      0.70 / 0.30   
...                                                                                ...   
Anomaly + mining IsolationForest benign-only fixed_window    0.70 / 0.30 (7216 / 3094)   
                                             time_delta      0.70 / 0.30 (5840 / 2504)   
                                             cscas_grouping  0.70 / 0.30 (4264 / 1828)   
                                             alertbert                     0.70 / 0.30   
                                             deepcase                      0.70 / 0.30   

                                                                F1  
Baseline         Model           Training    Grouping               
Internal system  LogReg          random      fixed_window    0.727  
                                             time_delta      0.028  
                                             cscas_grouping  0.135  
                                             alertbert       0.635  
                                             deepcase        0.659  
...                                                            ...  
Anomaly + mining IsolationForest benign-only fixed_window    0.397  
                                             time_delta      0.096  
                                             cscas_grouping  0.288  
                                             alertbert       0.840  
                                             deepcase        0.840  

[120 rows x 2 columns]

=== russellmitchell   (groupings: fixed_window, time_delta, cscas_grouping, alertbert, deepcase, 120 rows) ===


Split  \
Baseline         Model           Training    Grouping                                    
Internal system  LogReg          random      fixed_window    0.70 / 0.30 (3915 / 1678)   
                                             time_delta      0.70 / 0.30 (3372 / 1446)   
                                             cscas_grouping  0.70 / 0.30 (3150 / 1350)   
                                             alertbert                     0.70 / 0.30   
                                             deepcase                      0.70 / 0.30   
...                                                                                ...   
Anomaly + mining IsolationForest benign-only fixed_window    0.70 / 0.30 (3915 / 1678)   
                                             time_delta      0.70 / 0.30 (3372 / 1446)   
                                             cscas_grouping  0.70 / 0.30 (3150 / 1350)   
                                             alertbert                     0.70 / 0.30   
                                             deepcase                      0.70 / 0.30   

                                                                F1  
Baseline         Model           Training    Grouping               
Internal system  LogReg          random      fixed_window      NaN  
                                             time_delta        NaN  
                                             cscas_grouping    NaN  
                                             alertbert         NaN  
                                             deepcase        0.915  
...                                                            ...  
Anomaly + mining IsolationForest benign-only fixed_window    0.240  
                                             time_delta      0.119  
                                             cscas_grouping  0.081  
                                             alertbert       0.199  
                                             deepcase        0.193  

[120 rows x 2 columns]

=== shaw   (groupings: fixed_window, time_delta, cscas_grouping, 72 rows) ===


...

Split  \
Baseline         Model           Training       Grouping                                    
Internal system  LogReg          random         fixed_window    0.70 / 0.30 (8229 / 3528)   
                                                time_delta      0.70 / 0.30 (6794 / 2912)   
                                                cscas_grouping  0.70 / 0.30 (5651 / 2423)   
                                 class-weighted fixed_window    0.70 / 0.30 (8229 / 3528)   
                                                time_delta      0.70 / 0.30 (6794 / 2912)   
...                                                                                   ...   
Anomaly + mining OneClassSVM     benign-only    time_delta      0.70 / 0.30 (6794 / 2912)   
                                                cscas_grouping  0.70 / 0.30 (5651 / 2423)   
                 IsolationForest benign-only    fixed_window    0.70 / 0.30 (8229 / 3528)   
                                                time_delta      0.70 / 0.30 (6794 / 2912)   
                                                cscas_grouping  0.70 / 0.30 (5651 / 2423)   

                                                                   F1  
Baseline         Model           Training       Grouping               
Internal system  LogReg          random         fixed_window    0.004  
                                                time_delta      0.006  
                                                cscas_grouping  0.125  
                                 class-weighted fixed_window    0.000  
                                                time_delta      0.000  
...                                                               ...  
Anomaly + mining OneClassSVM     benign-only    time_delta      0.042  
                                                cscas_grouping  0.100  
                 IsolationForest benign-only    fixed_window    0.136  
                                                time_delta      0.043  
                                                cscas_grouping  0.037  

[72 rows x 2 columns]

=== wheeler   (groupings: fixed_window, time_delta, cscas_grouping, 72 rows) ===


Split  \
Baseline         Model           Training       Grouping                                     
Internal system  LogReg          random         fixed_window    0.70 / 0.30 (14686 / 6295)   
                                                time_delta      0.70 / 0.30 (11546 / 4949)   
                                                cscas_grouping  0.70 / 0.30 (12447 / 5335)   
                                 class-weighted fixed_window    0.70 / 0.30 (14686 / 6295)   
                                                time_delta      0.70 / 0.30 (11546 / 4949)   
...                                                                                    ...   
Anomaly + mining OneClassSVM     benign-only    time_delta      0.70 / 0.30 (11546 / 4949)   
                                                cscas_grouping  0.70 / 0.30 (12447 / 5335)   
                 IsolationForest benign-only    fixed_window    0.70 / 0.30 (14686 / 6295)   
                                                time_delta      0.70 / 0.30 (11546 / 4949)   
                                                cscas_grouping  0.70 / 0.30 (12447 / 5335)   

                                                                   F1  
Baseline         Model           Training       Grouping               
Internal system  LogReg          random         fixed_window    0.679  
                                                time_delta      0.071  
                                                cscas_grouping  0.066  
                                 class-weighted fixed_window    0.719  
                                                time_delta      0.059  
...                                                               ...  
Anomaly + mining OneClassSVM     benign-only    time_delta      0.062  
                                                cscas_grouping  0.095  
                 IsolationForest benign-only    fixed_window    0.458  
                                                time_delta      0.036  
                                                cscas_grouping  0.153  

[72 rows x 2 columns]

=== wilson   (groupings: fixed_window, time_delta, cscas_grouping, 72 rows) ===


Split  \
Baseline         Model           Training       Grouping                                     
Internal system  LogReg          random         fixed_window    0.70 / 0.30 (18567 / 7958)   
                                                time_delta      0.70 / 0.30 (14419 / 6180)   
                                                cscas_grouping  0.70 / 0.30 (10605 / 4545)   
                                 class-weighted fixed_window    0.70 / 0.30 (18567 / 7958)   
                                                time_delta      0.70 / 0.30 (14419 / 6180)   
...                                                                                    ...   
Anomaly + mining OneClassSVM     benign-only    time_delta      0.70 / 0.30 (14419 / 6180)   
                                                cscas_grouping  0.70 / 0.30 (10605 / 4545)   
                 IsolationForest benign-only    fixed_window    0.70 / 0.30 (18567 / 7958)   
                                                time_delta      0.70 / 0.30 (14419 / 6180)   
                                                cscas_grouping  0.70 / 0.30 (10605 / 4545)   

                                                                   F1  
Baseline         Model           Training       Grouping               
Internal system  LogReg          random         fixed_window    0.043  
                                                time_delta      0.049  
                                                cscas_grouping  0.117  
                                 class-weighted fixed_window    0.044  
                                                time_delta      0.049  
...                                                               ...  
Anomaly + mining OneClassSVM     benign-only    time_delta      0.040  
                                                cscas_grouping  0.130  
                 IsolationForest benign-only    fixed_window    0.343  
                                                time_delta      0.041  
                                                cscas_grouping  0.144  

[72 rows x 2 columns]

## Full experiment grid (LaTeX)

Exhaustive analogue of the CSCAS notebook's exhaustive summary table: **every**
`(baseline, model, features, scenario, grouping method, training condition)`
combination that has a result file, one row each. Reads every `ait_ads_*`
result JSON, ignoring the Settings cell, and reuses `PS_SPEC` / `_applicable` /
`_psload` / `PS_CONDITIONS` / `_FLAT_TRAINING` from the per-scenario section
above rather than redefining the same spec twice.

AIT-ADS keeps CSCAS's **Features** column (base schema vs. the
mining-augmented variant) but replaces its **Split** axis (one dataset,
varying eval-set size) with **Scenario** and **Grouping method** instead --
AIT-ADS's real second axis of variation, which CSCAS (pregrouped, one
dataset) doesn't have:

| knob | column |
|---|---|
| baseline family | **Baseline** (Internal system · Internal + mining · LLM (zero-shot/general/domain-specific) · Anomaly (+ mining)) |
| learner | **Model** (RF / LogReg / XGBoost / DistilBERT / SecureBERT 2.0 / Llama-3.1 (8B, 70B) / Qwen2.5 (7B, 72B) / OneClassSVM / IsolationForest) |
| input representation | **Features** (`Base (5)` reduced tabular schema · `Base (5) + mined` the same schema plus attribute-mined symbolic features · `tokens as text` for the text/LLM baselines) |
| scenario | **Scenario** (one of the 8 AIT-ADS scenarios) |
| grouping method | **Grouping** (`fixed_window` / `time_delta` / `cscas_grouping` for every scenario; `alertbert` / `deepcase` only for `fox` / `russellmitchell`) |
| training regime | **Training** (`random` / `class-weighted` for the trainable classifiers · `benign-only` for the anomaly detectors · `none` for zero-shot -- no `guided`, CSCAS-only) |

**Precision / Recall / F1** are mean$_{\pm\text{sd}}$ over 5 seeds for every
trainable row; the zero-shot rows and OneClassSVM's deterministic single fit
($\pm$0.000) have no seed spread. **AUC** (anomaly rows only, threshold-free) is
blank elsewhere. `russellmitchell` (every grouping but `deepcase`) has no row for the seven classifier methods -- an accepted exclusion, not a gap (every attack in its raw timeline falls in the last 30% regardless of grouping, so train is 100% benign -- see the intro cell); the four anomaly methods and zero-shot are exempt and do get a row there. `harrison`/`santos` don't appear in this table at all -- excluded from the notebook's aggregate reporting entirely, see the intro cell. All rows are scored on that scenario's full test split -- no `Full test` vs `Subsample` axis, unlike CSCAS. Writes `results/ait_ads_baseline_full_grid.tex`
as a `longtable` (this many rows won't fit one page -- landscape may help in
the final document).

In [29]:
# Exhaustive full-grid LaTeX table -- the AIT-ADS analogue of the CSCAS
# notebook's exhaustive summary table (see the markdown above for the column
# mapping). Reuses PS_SPEC / _applicable / _psload / PS_CONDITIONS /
# _FLAT_TRAINING / AIT_ADS_SCENARIOS from the per-scenario section above
# rather than redefining the same spec twice.

# Input representation, keyed by the same `baseline` label PS_SPEC assigns --
# mirrors the LaTeX-summary cell's per-entry `features` field (TABLE_SPEC)
# rather than reusing it directly, since PS_SPEC doesn't carry one.
FEATURES_LABEL_BY_BASELINE = {
    "Internal system": "Base (5)",
    "Internal + mining": "Base (5) + mined",
    "LLM (general)": "tokens as text",
    "LLM (domain-specific)": "tokens as text",
    "LLM (zero-shot)": "tokens as text",
    "Anomaly": "Base (5)",
    "Anomaly + mining": "Base (5) + mined",
}


def _pick_full(sub: dict | None) -> dict | None:
    """Precision/recall/f1 mean+-sd (None sd for a single run) plus auc, read
    off whichever dict actually carries them: for a trainable row `sub` is
    data[condition]; for a flat row `sub` is `data` itself. Same convention
    as _pick() in the LaTeX-summary cell above, plus `auc` (blank for every
    non-anomaly row, since only the anomaly result JSONs carry it)."""
    if sub is None:
        return None
    out = {m: sub.get(m) for m in ("precision", "recall", "f1")}
    out["auc"] = sub.get("auc")
    seeds = sub.get("seeds") or []
    zero_sd = sub.get("kind") == "anomaly"  # OneClassSVM: deterministic single fit
    for m in ("precision", "recall", "f1"):
        vals = [d[m] for d in seeds]
        out[f"{m}_sd"] = float(np.std(vals, ddof=1)) if len(vals) > 1 else (0.0 if zero_sd else None)
    return out


full_rows = []
gid = 0
for section, baseline, model, kind, tmpl in PS_SPEC:
    conds = [(None, _FLAT_TRAINING[section])] if kind == "flat" else PS_CONDITIONS
    features = FEATURES_LABEL_BY_BASELINE.get(baseline, "Base (5)")
    for s in AIT_ADS_SCENARIOS:
        for g in _applicable(s):
            data = _psload(tmpl.format(g=g, s=s))
            if data is None:
                continue  # no result file for this (baseline, scenario, grouping) -- omit, don't blank
            block = []
            for cond_key, cond_label in conds:
                sub = data if kind == "flat" else data.get(cond_key)
                m = _pick_full(sub)
                if m is None:
                    continue
                block.append({
                    "gid": gid, "section": section, "Baseline": baseline, "Model": model,
                    "Features": features, "Scenario": s, "Grouping": g, "Training": cond_label,
                    "Precision": m["precision"], "Precision_sd": m.get("precision_sd"),
                    "Recall": m["recall"], "Recall_sd": m.get("recall_sd"),
                    "F1": m["f1"], "F1_sd": m.get("f1_sd"),
                    "AUC": m.get("auc"),
                })
            if block:
                full_rows.extend(block)
                gid += 1

full_grid_df = pd.DataFrame(full_rows).drop(columns=["gid", "section"]).round(4) if full_rows else pd.DataFrame()
print(f"{len(full_rows)} rows across {gid} (baseline, model, features, scenario, grouping) blocks")


# --- emit LaTeX (repeated Baseline/Model/Features/Scenario/Grouping blanked within a block) ---
def _fg_cell(v, sd=None) -> str:
    if v is None or (isinstance(v, float) and pd.isna(v)):
        return "--"
    if sd is None or pd.isna(sd):
        return f"{v:.3f}"
    return f"${v:.3f}_{{\\pm {sd:.3f}}}$"


tex_lines = []
prev_section = prev_gid = None
for row in full_rows:
    if prev_section is not None and row["section"] != prev_section:
        tex_lines.append(r"\midrule")
    elif prev_gid is not None and row["gid"] != prev_gid:
        tex_lines.append(r"\addlinespace")
    first = row["gid"] != prev_gid
    prev_section, prev_gid = row["section"], row["gid"]
    b = row["Baseline"] if first else ""
    mo = row["Model"] if first else ""
    fe = row["Features"] if first else ""
    sc = row["Scenario"] if first else ""
    gr = row["Grouping"].replace("_", r"\_") if first else ""
    auc = "--" if row["AUC"] is None else f"{row['AUC']:.3f}"
    tex_lines.append(
        f"{b} & {mo} & {fe} & {sc} & {gr} & {row['Training']} & "
        f"{_fg_cell(row['Precision'], row['Precision_sd'])} & "
        f"{_fg_cell(row['Recall'], row['Recall_sd'])} & "
        f"{_fg_cell(row['F1'], row['F1_sd'])} & {auc}" + r" \\"
    )

latex = (
    r"""\begin{longtable}{llllllcccc}
\caption{Complete AIT-ADS baseline results: every (baseline, model, features,
scenario, grouping method, training condition) combination with a result
file. Trainable rows report mean$_{\pm\text{sd}}$ over 5 seeds: the zero-shot
rows and OneClassSVM (a deterministic single fit, $\pm$0.000) have no seed
spread. \textbf{Precision/recall/F1} are at each model's own decision
threshold; AUC (blank outside the anomaly rows) is the threshold-free
headline metric for those detectors, at the default rejection rate (nu /
contamination = 0.05).
\textbf{Features} is \texttt{Base (5)} (the reduced tabular schema),
\texttt{Base (5) + mined} (the same schema plus attribute-mined symbolic
features), or \texttt{tokens as text} for the text/LLM baselines.
The \textbf{Training} column reports the training-pool condition:
\texttt{random} = random undersampling of the majority (irrelevant) class down
to the minority (important) count, \texttt{class-weighted} = the full
natural-ratio training pool with balanced class weights
(\texttt{scale\_pos\_weight} for XGBoost), \texttt{benign-only} = the anomaly
detectors, fit on benign rows only, no attack examples used in training,
\texttt{none} = zero-shot, no training step -- no \texttt{guided} condition,
CSCAS-only (AIT-ADS has no SCAS-equivalent outlier signal to guide sampling
with). \textbf{Grouping} is one of \texttt{fixed\_window} / \texttt{time\_delta}
/ \texttt{cscas\_grouping} (every scenario) or \texttt{alertbert} /
\texttt{deepcase} (\texttt{fox} / \texttt{russellmitchell} only -- the
scenarios held out of the learned groupers' training). Covers 6 of AIT-ADS's 8
scenarios -- \texttt{harrison}/\texttt{santos} are excluded from this table
entirely (see the intro cell). \texttt{russellmitchell} (every grouping but
\texttt{deepcase}) has no row for the seven classifier methods -- every attack
in its raw timeline falls in the last 30\% regardless of grouping, so train is
100\% benign; the four anomaly methods and zero-shot are exempt (see the intro
cell). All rows are scored on that scenario's full test split -- no `Full
test' vs `Subsample' split axis: unlike CSCAS, no frozen eval subsample is
used.}
\label{tab:ait-ads-baseline-results-full} \\
\toprule
\textbf{Baseline} & \textbf{Model} & \textbf{Features} & \textbf{Scenario} & \textbf{Grouping} & \textbf{Training} & \textbf{Precision} & \textbf{Recall} & \textbf{F1} & \textbf{AUC} \\
\midrule
\endfirsthead
\multicolumn{10}{c}{\tablename\ \thetable{} -- continued from previous page} \\
\toprule
\textbf{Baseline} & \textbf{Model} & \textbf{Features} & \textbf{Scenario} & \textbf{Grouping} & \textbf{Training} & \textbf{Precision} & \textbf{Recall} & \textbf{F1} & \textbf{AUC} \\
\midrule
\endhead
\midrule \multicolumn{10}{r}{\textit{continued on next page}} \\
\endfoot
\bottomrule
\endlastfoot
"""
    + "\n".join(tex_lines)
    + r"""
\end{longtable}
"""
)

latex_path = RESULTS_DIR / "ait_ads_baseline_full_grid.tex"
latex_path.write_text(latex, encoding="utf-8")
print(f"LaTeX table written to {latex_path}")
full_grid_df


464 rows across 320 (baseline, model, features, scenario, grouping) blocks
LaTeX table written to /Users/annavisman/stack/TUDelft/thesis/msc-thesis/src/thesis/baselines/results/ait_ads_baseline_full_grid.tex


,Baseline,Model,Features,Scenario,Grouping,Training,Precision,Precision_sd,Recall,Recall_sd,F1,F1_sd,AUC
0,Internal system,LogReg,Base (5),fox,fixed_window,random,0.8073,0.0940,0.6646,0.0037,0.7270,0.0350,NaN
1,Internal system,LogReg,Base (5),fox,fixed_window,class-weighted,0.8580,0.0000,0.6681,0.0000,0.7512,0.0000,NaN
2,Internal system,LogReg,Base (5),fox,time_delta,random,0.0145,0.0010,0.2700,0.0209,0.0275,0.0018,NaN
3,Internal system,LogReg,Base (5),fox,time_delta,class-weighted,0.0138,0.0000,0.2500,0.0000,0.0262,0.0000,NaN
4,Internal system,LogReg,Base (5),fox,cscas_grouping,random,0.0782,0.0184,0.5223,0.0135,0.1352,0.0262,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...
459,Anomaly + mining,IsolationForest,Base (5) + mined,wheeler,time_delta,benign-only,0.0205,0.0050,0.1576,0.0395,0.0363,0.0089,0.6930
460,Anomaly + mining,IsolationForest,Base (5) + mined,wheeler,cscas_grouping,benign-only,0.1399,0.0167,0.1692,0.0060,0.1528,0.0110,0.5271
461,Anomaly + mining,IsolationForest,Base (5) + mined,wilson,fixed_window,benign-only,0.4148,0.0803,0.2955,0.1074,0.3434,0.0987,0.8133
462,Anomaly + mining,IsolationForest,Base (5) + mined,wilson,time_delta,benign-only,0.0316,0.0093,0.0584,0.0166,0.0410,0.0119,0.5196
